In [90]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MEAO = pd.read_excel('G:\Priyabrata\ML\Material data repository.xlsx')
INPUT=MEAO.iloc[:,2:11]  #all INPUT (Variables)
TARGET=MEAO.iloc[:,20]   #all PHASES SS,SS+IM,AM
#TARGET = MEAO.iloc[:,19]

Y=MEAO.iloc[:,22] #coded phases. 0:SS, 1:SS+IM, 2:AM
#Y = MEAO.iloc[:,23] #coded phases. 0:MSS, 1:SSS, 2:SS+IM, 3:AM


selected_INPUT = INPUT.drop(['Mixing entropy','γ','VEC','Pauling electronegativity','Λ','Molar volume dispersity'],axis=1)
#selected_INPUT = INPUT.drop(['Mixing entropy','γ','Pauling electronegativity','Molar volume dispersity'],axis=1)
selected_INPUT
selected_INPUT

#normalized_INPUT=(selected_INPUT-selected_INPUT.mean())/selected_INPUT.std()
selected_INPUT.std()
normalized_INPUT=(selected_INPUT-selected_INPUT.min())/(selected_INPUT.max()-selected_INPUT.min())
normalized_INPUT

,Mixing enthalpy,Ω,Atomic size mismatch
0,0.514393,0.065497,0.235336
1,0.676546,0.069756,0.225748
2,0.705331,0.080604,0.213648
3,0.734628,0.092112,0.198033
4,0.764265,0.105445,0.178217
...,...,...,...
241,0.743485,0.161006,0.198873
242,0.741611,0.162020,0.207589
243,0.739908,0.162491,0.215799
244,0.705672,0.127395,0.064935


In [2]:
#Repeated Stratified k fold cross validation (for imbalanced datasets)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from numpy import mean
from numpy import std
from sklearn.model_selection import cross_val_score


    
nskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=1)
model2 = MLPClassifier(hidden_layer_sizes=(10,10), activation='relu', solver='sgd', alpha=0.0000001, learning_rate='adaptive', learning_rate_init=0.1,max_iter=15000, shuffle=True, warm_start='False') #batch size has no meaning for lbfgs
    
# evaluate model
scores = cross_val_score(model2, normalized_INPUT, Y, scoring='f1_macro', cv=nskf, verbose=3,n_jobs=-1)
scores
print('f1 score: %.3f (%.3f)' % (mean(scores), std(scores)))
print('\nMaximum f1 score: ', max(scores))
print('\nMinimum f1 score: ', min(scores))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    3.4s


f1 score: 0.875 (0.051)

Maximum f1 score:  0.9796592931604374

Minimum f1 score:  0.7520255413993554


[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    5.8s finished


In [91]:
All_MEA = pd.read_excel('G:\Priyabrata\Optimization\Mg60 optimization.xlsx','Sheet5')
Quarternary_MEA = All_MEA.iloc[:,1:4]
normalized_MEA=(Quarternary_MEA-Quarternary_MEA.mean())/Quarternary_MEA.std()
#normalized_MEA=(Quarternary_MEA-selected_INPUT.mean())/selected_INPUT.std()
model2.fit(normalized_INPUT,Y)
Predicted=model2.predict(normalized_MEA)   #giving accuracy
Predicted
normalized_MEA

,Mixing enthalpy,Omega,Atomic size mismatch
0,-0.842627,-0.465948,-1.361181
1,-0.400232,0.026380,-0.892694
2,-0.082221,1.530209,0.160026
3,-0.628172,-0.389404,-0.986831
4,0.120185,-0.162209,0.963061
...,...,...,...
4840,-1.561592,-0.633420,-0.079496
4841,-2.730187,-0.698678,-0.514952
4842,-0.397183,0.077709,0.273631
4843,-1.793971,-0.655090,-0.162891


In [92]:
df = pd.DataFrame(Predicted, columns = ['Prediction'])
#modulus = All_MEA.iloc[:,5]
  
df['Youngs modulus'] = All_MEA.iloc[:,4]
print(df)

      Prediction  Youngs modulus
0              0          83.625
1              0         111.000
2              0          64.500
3              0          89.250
4              2          62.925
...          ...             ...
4840           2          81.250
4841           2          47.475
4842           2          78.975
4843           2          69.975
4844           2          71.225

[4845 rows x 2 columns]


In [93]:
df1=df[df['Prediction'] == 0]
df1

,Prediction,Youngs modulus
0,0,83.625
1,0,111.000
2,0,64.500
3,0,89.250
5,0,70.250
...,...,...
4817,0,180.250
4820,0,124.000
4821,0,155.500
4824,0,112.725


In [16]:
df2 = df1[df1['Youngs modulus'] < 75]
df2








,Prediction,Youngs modulus
2,0,64.500
5,0,70.250
14,0,68.250
16,0,65.975
18,0,61.625
...,...,...
4514,0,64.225
4627,0,74.225
4629,0,65.225
4632,0,66.475


In [181]:
All_MEA = pd.read_excel('G:\Priyabrata\Optimization\Mg60 optimization.xlsx','Sheet5')
Quarternary_MEA = All_MEA.iloc[:,1:4]
#normalized_MEA=(Quarternary_MEA-Quarternary_MEA.mean())/Quarternary_MEA.std()
#normalized_MEA=(Quarternary_MEA-[-12.08,5.60,5.4178])/[13.261,6.398,3.654]
#normalized_MEA=(Quarternary_MEA-Quarternary_MEA.min())/([10,20,25.86]-Quarternary_MEA.min())
from sklearn import preprocessing

scaler = preprocessing.MinMaxScaler(feature_range=(-1, 1))  #set feature range and see the change in coeff matrix. Not changing with range. Same as (INPUT-INPUT.mean())/INPUT.std()
names = selected_INPUT.columns
d = scaler.fit_transform(selected_INPUT)      #fit transform gives arrays not panda dataframe. same for unscaling
scaled_INPUT = pd.DataFrame(d, columns=names)

name =Quarternary_MEA.columns
e = scaler.fit_transform(Quarternary_MEA) 
scaled_MEA = pd.DataFrame(e, columns=name)

model2.fit(scaled_INPUT,Y)
Predicted=model2.predict(scaled_MEA)   #giving accuracy

df = pd.DataFrame(Predicted, columns = ['Prediction'])
df['Mixing enthalpy'] = All_MEA.iloc[:,1]
df['Omega'] = All_MEA.iloc[:,2]
df['Atomic size mismatch'] = All_MEA.iloc[:,3]
  
df['Youngs modulus'] = All_MEA.iloc[:,4]
df1=df[df['Prediction'] == 0]
df1


,Prediction,Mixing enthalpy,Omega,Atomic size mismatch,Youngs modulus
5,0,2.51600,7.223034,5.527067,70.250
7,0,1.56400,12.614580,5.527067,106.500
8,0,-4.31000,4.690531,6.501834,177.250
9,0,0.00800,20.000000,8.086515,121.250
10,0,5.64500,3.293864,6.068509,142.500
...,...,...,...,...,...
4785,0,20.91325,1.078872,4.000000,200.750
4790,0,0.64925,20.000000,15.521072,144.250
4809,0,-0.77450,19.850665,17.426002,115.225
4812,0,-0.84400,20.000000,14.745925,188.000


In [182]:
df2 = df1[df1['Youngs modulus'] < 75]
df2
df3 = df2[df2['Atomic size mismatch'] < 6]
df3

,Prediction,Mixing enthalpy,Omega,Atomic size mismatch,Youngs modulus
5,0,2.51600,7.223034,5.527067,70.250
94,0,4.38975,3.676465,5.671201,61.000
157,0,12.99450,1.716813,4.234528,71.375
230,0,12.24300,1.612014,4.136895,65.000
350,0,5.54400,3.657060,4.796842,62.125
755,0,8.43850,2.531727,4.648343,73.500
3617,0,4.56400,3.192206,5.995437,29.400
